# 06 — Deploy the Fake Review Checker website (free static Hugging Face Space)

Converts the trained models for in-browser use and publishes `web/` as a **static** Hugging Face Space (free):
`https://huggingface.co/spaces/<your-username>/fake-review-checker`

- **BERT / XLNet** → ONNX (torch dynamo exporter) → int8 dynamic quantisation (~4× smaller)
- **CNN** → raw weights (`weights.bin` + `manifest.json`); its forward pass is re-implemented in `web/app.js`
- A parity check compares the converted models with the originals before uploading

Run on a **CPU** runtime (no GPU needed). The login cell shows a code to authorise at hf.co/oauth/device.

In [ ]:
# --- Setup: mount Google Drive (dataset lives there, not in GitHub) ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/FakeReviewProject'   # change if your folder differs
DATA_DIR    = f'{PROJECT_DIR}/dataset'                     # train.csv, test.csv

# True  = train on a LENGTH-BALANCED set (equal fake/genuine in every review-length band), so review length
#         alone cannot decide the label; models and results go to outputs_lengthbal/
# False = original setup (full imbalanced training set); results in outputs/
BALANCE_BY_LENGTH = False   # set True to reproduce the length-balanced experiment
OUT_DIR     = f'{PROJECT_DIR}/outputs_lengthbal' if BALANCE_BY_LENGTH else f'{PROJECT_DIR}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR), '->', OUT_DIR)

In [ ]:
!pip -q install -U huggingface_hub onnx onnxruntime onnxscript
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import json, shutil, numpy as np
STAGE = '/content/web_stage'
shutil.rmtree(STAGE, ignore_errors=True)
!git clone -q --depth 1 https://github.com/AjayRaju-18/ShreyaProject.git /content/repo_src
shutil.copytree('/content/repo_src/web', STAGE)
os.makedirs(f'{STAGE}/models', exist_ok=True)

SAMPLES = [  # (text, title, rating) — used for the parity check and to verify the website afterwards
    ('Absolutely amazing product!!! Best purchase ever, everyone should buy this, five stars!!!', 'Amazing', 5),
    ('Case fits my phone well. The buttons are a bit stiff but it has survived two drops so far.', 'Decent case', 4),
    ('Stopped working after a week. Waste of money.', 'Broke quickly', 1),
]
REFERENCE = {}

## CNN → weights for the JavaScript forward pass

In [ ]:
import keras, pickle
from keras import layers as KL
cnn = keras.models.load_model(f'{OUT_DIR}/cnn_model.keras', compile=False)
cfg = json.load(open(f'{OUT_DIR}/cnn_config.json'))
scaler = pickle.load(open(f'{OUT_DIR}/cnn_scaler.pkl', 'rb'))

convs  = [l for l in cnn.layers if isinstance(l, KL.Conv1D)]
denses = [l for l in cnn.layers if isinstance(l, KL.Dense)]
emb    = [l for l in cnn.layers if isinstance(l, KL.Embedding)][0]
named = {'embedding': emb.get_weights()[0]}
for i, l in enumerate(convs, 1):
    named[f'conv{i}_kernel'], named[f'conv{i}_bias'] = l.get_weights()
for i, l in enumerate(denses, 1):
    named[f'dense{i}_kernel'], named[f'dense{i}_bias'] = l.get_weights()

os.makedirs(f'{STAGE}/models/cnn', exist_ok=True)
manifest, off, blobs = {}, 0, []
for k, v in named.items():
    v = np.ascontiguousarray(v, dtype='<f4'); manifest[k] = {'offset': off, 'shape': list(v.shape)}
    off += v.size; blobs.append(v.ravel())
np.concatenate(blobs).tofile(f'{STAGE}/models/cnn/weights.bin')
json.dump(manifest, open(f'{STAGE}/models/cnn/manifest.json', 'w'))
json.dump(json.load(open(f'{OUT_DIR}/cnn_vocab.json')), open(f'{STAGE}/models/cnn/vocab.json', 'w'))
json.dump({'max_len': cfg['max_len'], 'threshold': cfg['threshold'], 'num_features': cfg['num_features'],
           'scaler_mean': scaler.mean_.tolist(), 'scaler_scale': scaler.scale_.tolist()},
          open(f'{STAGE}/models/cnn/config.json', 'w'), indent=2)
print({k: v['shape'] for k, v in manifest.items()})

# Reference probabilities from the real Keras model (same preprocessing as notebook 05)
!pip -q install nltk vaderSentiment
import re, pandas as pd
from nltk.tokenize import TweetTokenizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader, tok = SentimentIntensityAnalyzer(), TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
URL, HTML, NONTXT = re.compile(r'https?://\S+|www\.\S+'), re.compile(r'<[^>]+>'), re.compile(r"[^a-z0-9'!?.,:;()\s]")
clean = lambda t: NONTXT.sub(' ', HTML.sub(' ', URL.sub(' ', str(t).lower())))
vocab = json.load(open(f'{OUT_DIR}/cnn_vocab.json'))
for text, title, rating in SAMPLES:
    ids = [vocab.get(w, 1) for w in tok.tokenize(clean(title + ' . ' + text))][:cfg['max_len']]
    X = np.zeros((1, cfg['max_len']), 'int32'); X[0, :len(ids)] = ids
    rs = vader.polarity_scores(text)['compound']
    f = pd.DataFrame([{'rating': rating, 'review_length': len(text.split()), 'review_sentiment': rs,
                       'summary_sentiment': vader.polarity_scores(title)['compound'],
                       'rating_sentiment_difference': rating / 5 - rs}])[cfg['num_features']]
    p = float(cnn.predict([X, scaler.transform(f).astype('float32')], verbose=0)[0, 0])
    REFERENCE.setdefault(text[:40], {})['CNN'] = round(p, 4)
print(REFERENCE)

## BERT / XLNet → ONNX (int8) with parity check

In [ ]:
import torch, onnx, onnxruntime as ort
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from onnxruntime.quantization import quantize_dynamic, QuantType

class Logits(torch.nn.Module):
    """Single logits output; float attention mask keeps XLNet exportable."""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask, token_type_ids):
        return self.m(input_ids=input_ids, attention_mask=attention_mask.float(), token_type_ids=token_type_ids).logits

NAMES = ['input_ids', 'attention_mask', 'token_type_ids']
def encode(tk, text, title, L):
    e = tk([title + ' . ' + text], truncation=True, max_length=L, return_tensors='pt')
    return e['input_ids'], e['attention_mask'], torch.zeros_like(e['input_ids'])

for name in ['BERT', 'XLNet']:
    src = f'{OUT_DIR}/{name.lower()}_model'
    if not os.path.exists(src):
        print('skip', name); continue
    pcfg = json.load(open(f'{src}/predict_config.json'))
    tk = AutoTokenizer.from_pretrained(src)
    w = Logits(AutoModelForSequenceClassification.from_pretrained(src)).eval()
    dst = f'{STAGE}/models/{name.lower()}'; os.makedirs(dst, exist_ok=True)
    fp32 = f'/content/{name}_fp32.onnx'
    seq = torch.export.Dim('seq', min=2, max=512)
    torch.onnx.export(w, encode(tk, *SAMPLES[0][:2], pcfg['max_len']), fp32, input_names=NAMES,
                      output_names=['logits'], dynamo=True, opset_version=18,
                      dynamic_shapes={k: {1: seq} for k in NAMES})
    m = onnx.load(fp32); del m.graph.value_info[:]; onnx.save(m, fp32 + '.clean.onnx')   # drop stale shapes
    quantize_dynamic(fp32 + '.clean.onnx', f'{dst}/model.onnx', weight_type=QuantType.QInt8)
    tk.save_pretrained(dst)
    json.dump({'max_len': pcfg['max_len'], 'threshold': pcfg['threshold']}, open(f'{dst}/config.json', 'w'), indent=2)

    sess = ort.InferenceSession(f'{dst}/model.onnx')
    for text, title, _ in SAMPLES:
        a = encode(tk, text, title, pcfg['max_len'])
        with torch.no_grad():
            p_torch = torch.softmax(w(*a), -1)[0, 1].item()
        lg = sess.run(None, {k: v.numpy() for k, v in zip(NAMES, a)})[0][0]
        p_onnx = float(np.exp(lg[1]) / np.exp(lg).sum())
        REFERENCE.setdefault(text[:40], {})[name] = round(p_torch, 4)
        print(f'{name:5s} torch={p_torch:.4f} onnx-int8={p_onnx:.4f} diff={abs(p_torch - p_onnx):.4f} | {text[:40]}')
json.dump(REFERENCE, open(f'{STAGE}/models/reference.json', 'w'), indent=2)
!du -sh {STAGE}/models/*

## Upload to a free static Space

In [ ]:
from huggingface_hub import HfApi
SPACE_NAME = 'fake-review-checker'
api = HfApi()
repo_id = f"{api.whoami()['name']}/{SPACE_NAME}"
api.create_repo(repo_id, repo_type='space', space_sdk='static', exist_ok=True)
api.upload_folder(folder_path=STAGE, repo_id=repo_id, repo_type='space', commit_message='Deploy Fake Review Checker')
print(f'Website: https://huggingface.co/spaces/{repo_id}')
print(f'Direct:  https://{repo_id.replace("/", "-").replace("_", "-").lower()}.static.hf.space')